# Portfolio Optimization Demo

Demonstrates return computation, risk analytics, minimum-variance / maximum-Sharpe / target-return optimization, efficient frontier, and Monte Carlo simulation.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.portfolio.returns import simple_returns, annualized_return, annualized_volatility
from src.portfolio.risk import sharpe_ratio, max_drawdown, correlation_matrix, covariance_matrix
from src.portfolio.optimization import minimum_variance_portfolio, maximum_sharpe_portfolio, target_return_portfolio, monte_carlo_simulation
from src.portfolio.efficient_frontier import efficient_frontier
from src.visualization.charts import plot_cumulative_returns, plot_efficient_frontier, plot_correlation_heatmap, plot_drawdown

# --- Generate synthetic price history (or load CSV) ---
np.random.seed(42)
dates = pd.date_range("2018-01-01", periods=1260, freq="B")  # ~5y daily
tickers = ["AAPL", "MSFT", "JPM", "XOM", "TLT"]
mus    = np.array([0.0006, 0.0005, 0.0003, 0.0002, 0.0001])
vols   = np.array([0.015, 0.013, 0.014, 0.012, 0.006])
prices = pd.DataFrame(index=dates)
for t, mu, vol in zip(tickers, mus, vols):
    prices[t] = 100 * np.cumprod(1 + np.random.normal(mu, vol, len(dates)))
prices.head()

In [ ]:
rets = simple_returns(prices)
rets.describe().round(4)

In [ ]:
ann_ret = annualized_return(rets)
ann_vol = annualized_volatility(rets)
pd.DataFrame({"Annualized Return": ann_ret, "Volatility": ann_vol}).round(4)

In [ ]:
# Risk tables
print("Sharpe (rf=2%):", sharpe_ratio(rets, risk_free_rate=0.02).round(3).to_dict())
print("Max DD AAPL:", f"{max_drawdown(prices=prices['AAPL']):.1%}")
correlation_matrix(rets).round(2)

In [ ]:
plot_cumulative_returns(prices=prices, save_path="../outputs/cumulative_returns.png")
plot_correlation_heatmap(correlation_matrix(rets), save_path="../outputs/correlation.png")

In [ ]:
# --- Optimization inputs: historical mean & covariance (annualized) ---
er  = annualized_return(rets)
cov = covariance_matrix(rets)  # annualized
er, cov.head()

In [ ]:
min_var = minimum_variance_portfolio(er, cov, risk_free_rate=0.02)
max_sh  = maximum_sharpe_portfolio(er, cov, risk_free_rate=0.02)
target  = target_return_portfolio(er, cov, target_return=0.10, risk_free_rate=0.02)

for name, res in [("Min Var", min_var), ("Max Sharpe", max_sh), ("Target 10%", target)]:
    print(f"\n{name}: Ret={res['expected_return']:.2%} Vol={res['volatility']:.2%} Sharpe={res['sharpe_ratio']:.2f}")
    print(res["weights"].round(3).to_dict())

In [ ]:
# Efficient frontier
frontier = efficient_frontier(er, cov, n_points=30, risk_free_rate=0.02)
frontier.head().round(4)

In [ ]:
# Monte Carlo cloud
mc = monte_carlo_simulation(er, cov, n_portfolios=3000, risk_free_rate=0.02, seed=42)
mc.head()

In [ ]:
plot_efficient_frontier(frontier, min_var=min_var, max_sharpe=max_sh, mc_df=mc, save_path="../outputs/efficient_frontier.png")

In [ ]:
# Drawdown of max-Sharpe portfolio (reconstruct weighted returns)
# Compute portfolio return series
weights = max_sh["weights"]
port_rets = (rets * weights).sum(axis=1)
plot_drawdown(returns=port_rets, title="Max-Sharpe Portfolio Drawdown", save_path="../outputs/drawdown.png")